# Problems 8 - Probabilities

Every problem has two parts:

* **Part (a)**: the classical textbook version. Answer it with a derivation (definition of conditional probability, Bayes' theorem). You can check it with a simulation.
* **Part (b)**: the same idea in a **smart city** setting. In some of them you will estimate the probabilities from a real data set instead of assuming them.

Write your answers in the text cells and your code in the code cells. Explain each step, not only the final number.

### Data for part (b): Bicing stations

A **snapshot** of every Bicing station in Barcelona, taken at one single moment (the time is in the `snapshot_utc` column). Source: Bicing's public GBFS feeds. Each row is a station.

| Column | Meaning |
|---|---|
| `station_id`, `name` | Station |
| `district` | District of Barcelona |
| `lat`, `lon` | Location |
| `capacity` | Number of docks |
| `bikes_available` | Bikes that can be taken **at that moment** |
| `docks_available` | Free docks at that moment |
| `status` | `IN_SERVICE`, `PLANNED`... |

We call a station **empty** if it has `bikes_available == 0`. Remember that this is one moment of one day: it is *observed data*, not the typical behaviour of the system.

In [ ]:
import math
import random

import numpy as np
import pandas as pd

BASE = "https://raw.githubusercontent.com/cienciadadesdsuab/DS26-27/main/Problemes/Session%208%20-%20Problems/"

bicing = pd.read_csv(BASE + "bicing_snapshot_2026-11-24.csv")
bicing["is_empty"] = bicing["bikes_available"] == 0

print("Snapshot (UTC):", bicing["snapshot_utc"].iloc[0], "| stations:", len(bicing))
bicing.head()

# Problem 1: Lottery

## (a) Two urns

We are given two urns.

* Urn A contains four balls: one black, two white and one pink.
* Urn B contains four hundred balls: two hundred black, 50 yellow, 40 cyan, 30 white, 25 green, 25 silver, 20 gold, and 10 purple.

One of the urns is selected **at random** and one ball is drawn. The ball is black.

**Q:** What is the probability that the urn where the ball was drawn from is urn A?

---

*Your Answer Here*

---

## (b) Which district?

We pick one of two districts **at random** (each with probability 1/2), and then one of its stations at random. The station we picked is **empty**.

Take the districts `08-NouBarris` (call it N) and `02-Eixample` (call it E).

**Q1.** Estimate from the snapshot the probability that a station is empty *given* its district, for each of the two districts.

**Q2.** Given that the station we picked is empty, what is the probability that it belongs to Nou Barris? Use Bayes' theorem.

**Q3.** Compare your answer with $P(N) = 1/2$. What does the comparison tell you about empty stations?

**Q4.** What changes in the reasoning if the districts are not chosen with equal probability, but in proportion to their number of stations?

In [ ]:
# your code here

---

*Your Answer Here*

---

# Problem 2: Children gender

## (a) Two children

Assume a family with two children. We assume that:

1. Each child is equally likely to be a boy or a girl.
2. The gender of the second child is independent of the gender of the first child.

**Q1.** What is the probability of each of the following scenarios?

a) No girls (only boys)

b) One girl, one boy

c) Two girls (no boys)

**Q2.** What is the probability of both children being girls if the first child is a girl?

**Q3.** What is the probability of both children being girls if at least one of the children is a girl?

---

*Your Answer Here*

---

In [ ]:
# (optional) check your answers with a simulation
import random
random.seed(0)

## (b) Two neighbouring stations

Now the "children" are **two neighbouring Bicing stations**, and "girl" is replaced by "the station is **empty**". The code below pairs every station with its nearest neighbour (a pair appears only once). In each pair, call the station with the lower index the *first* one.

In [ ]:
# Pair every station with its nearest neighbour (distance on the sphere, in km)
lat = np.radians(bicing["lat"].values)
lon = np.radians(bicing["lon"].values)
dlat = lat[:, None] - lat[None, :]
dlon = lon[:, None] - lon[None, :]
a = np.sin(dlat / 2) ** 2 + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin(dlon / 2) ** 2
dist_km = 2 * 6371 * np.arcsin(np.sqrt(a))
np.fill_diagonal(dist_km, np.inf)

nearest = dist_km.argmin(axis=1)
pairs = sorted({tuple(sorted((i, j))) for i, j in enumerate(nearest)})

empty = bicing["is_empty"].values
first_empty = np.array([empty[i] for i, j in pairs])    # event A: the first station is empty
second_empty = np.array([empty[j] for i, j in pairs])   # event B: the second station is empty

print("pairs of neighbouring stations:", len(pairs))
print("median distance inside a pair:", round(np.median([dist_km[i, j] for i, j in pairs]) * 1000), "m")

**Q1.** Estimate $P(A)$, $P(B)$ and $P(A, B)$ from the pairs. If the two stations were independent, what would $P(A, B)$ be? Is it? Also compute, as in part (a), the probabilities of "no empty", "one empty" and "two empty" stations in a pair.

**Q2.** What is the probability that both stations are empty given that the first is empty?

**Q3.** What is the probability that both are empty given that at least one is empty?

**Q4.** Compare Q2 and Q3 with your results in part (a). What is the same and what is different? Why is independence not a reasonable assumption for neighbouring stations? (Think about what causes a station to be empty.)

In [ ]:
# your code here

---

*Your Answer Here*

---

# Problem 3: Three fair coins

## (a) Three tosses

You toss a fair coin three times.

**Q1.** What is the probability of three heads?

**Q2.** What is the probability that you observe exactly one heads?

**Q3.** Given that you have observed at least one heads, what is the probability that you observe at least two heads?

---

*Your Answer Here*

---

In [ ]:
# (optional) check your answers with a simulation
import random
random.seed(0)

## (b) Three random stations

We pick **three Bicing stations at random** (each one independently of the others, so the same station could in principle appear twice). The "head" is now "the station is **empty**", but the coin is **not fair**: the probability of an empty station is the share of empty stations in the snapshot, which we call $p$.

**Q1.** Compute $p$ from the snapshot.

**Q2.** What is the probability that all three stations are empty?

**Q3.** What is the probability that exactly one of the three is empty?

**Q4.** Given that at least one of the three is empty, what is the probability that at least two are empty?

**Q5.** Check Q2-Q4 with a simulation that picks three random stations from the snapshot 10,000 times. Do the simulated values match your calculations? Why should they?

**Q6.** Would the same calculation be valid for **three neighbouring stations**? Use your conclusion in Problem 2 to argue what you would expect to change (higher or lower probability of three empty stations, and why). You do not need to compute it.

In [ ]:
# your code here

---

*Your Answer Here*

---

# Problem 4: Fake coin

## (a) The two-headed coin

A box contains four coins, three regular coins and one fake two-headed coin.

**Q1.** You pick a coin at random and toss it. What is the probability that it lands tails up?

**Q2.** You pick a coin at random and toss it, and get heads. What is the probability that it is the two-headed coin?

---

*Your Answer Here*

---

In [ ]:
# (optional) check your answers with a simulation
import random
random.seed(0)

## (b) The stuck sensor

A city has **100 air-quality sensors**. Two of them are **stuck**: they raise an alert every hour, whatever the air is like. The other 98 work properly and raise a (false) alert in a given hour with probability **0.05**, independently from hour to hour.

**Q1.** You pick a sensor at random. What is the probability that it raises **no alert** in the next hour?

**Q2.** You pick a sensor at random and it raises an alert in one hour. What is the probability that it is one of the stuck sensors?

**Q3.** The same sensor raises an alert in **two consecutive hours**. What is the probability that it is stuck? (Think: what changes between Q2 and Q3?)

**Q4.** After how many consecutive alerts would you be more than 99% sure that the sensor is stuck?

**Q5.** Compare with the disease example from the lab: what do the two situations have in common, and why is the answer to Q2 so much lower than the "reliability" of the sensor might suggest?

In [ ]:
# your code here

---

*Your Answer Here*

---

---

*Data attribution (Problems 1-3, part (b)): Bicing station status and information, GBFS feeds of the Bicing service (Barcelona). Source: Ajuntament de Barcelona, licence CC BY. One snapshot, taken at the time indicated in the file.*